# Image Classification Model using CNN [Tutorial]

### Notebook Spine
- Load and prepare CIFAR-10 data
- Build baseline MLP models
- Build a CNN model
- Evaluate model performance
- Visualize predictions

### Problem Statement

Use CIFAR10 data to build image classification models (MLP and CNN).

### Load and prepare data for modeling

In [ ]:
# labels
class_labels = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
                'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
from tensorflow.keras.datasets import cifar10

(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = cifar10.load_data()

X_train_raw.shape, y_train_raw.shape, X_test_raw.shape, y_test_raw.shape

In [ ]:
# reshape the target labels into vector form
y_train_raw = y_train_raw.reshape(-1, )
y_test_raw = y_test_raw.reshape(-1, )

y_train_raw.shape, y_test_raw.shape

In [ ]:
# create dummies for the target labels
from tensorflow.keras.utils import to_categorical

y_train = to_categorical(y_train_raw, 10)
y_test = to_categorical(y_test_raw, 10)

y_train.shape, y_test.shape

In [ ]:
# scale pixel values to [0,1]
X_train = X_train_raw.astype('float32') / 255.0
X_test = X_test_raw.astype('float32') / 255.0

X_train.shape, y_train.shape, X_test.shape, y_test.shape

* Note: Inputs are scaled to `[0,1]` by dividing pixel values by `255`.

For each lable (target value), we have equal number of records (5,000).

### Visualize data

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 4, 
                         figsize=(6, 6),
                         subplot_kw={'xticks':[], 'yticks':[]})

for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_train_raw[i])
    
    act = class_labels[y_train_raw[i]]
    ax.text(0.05, 0.05, act, color='white', fontsize=10,
            weight='semibold', transform=ax.transAxes)

plt.show();

### Build a NN model with hidden layers

Before we start, we need to take a small digression. The keras Neural Network model training results are not easily reproducible since it involves a lot of shuffling and random initializations. In order to maintain consitency, we will have to initialize some random seeds before every model run. We will create a function to do this.

In [ ]:
from tensorflow import random as tf_random
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Flatten, Conv2D, MaxPooling2D

import numpy as np
import random
import seaborn as sns


def init_seeds(s):
    '''
    Initializes random seeds prior to model training 
    to ensure reproducibality of training results.
    '''
    tf_random.set_seed(s)
    np.random.seed(s)
    random.seed(s)


def print_metrics(model, X, y, batch_size=1):
    loss, accuracy = model.evaluate(X, y, batch_size=batch_size, verbose=0)
    print(f'Loss: {loss:.4f}, Accuracy: {accuracy*100:.2f}%')
    return loss, accuracy


def predict_classes_from_probs(probs):
    return np.argmax(probs, axis=1)


def predict_classes(model, X):
    probs = model.predict(X)
    return np.argmax(probs, axis=1)


def plot_confusion_matrix(y_true, y_pred, title, figsize=(10, 10), cmap='YlGnBu'):
    from sklearn.metrics import confusion_matrix

    cfm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=figsize)
    sns.heatmap(cfm, annot=True, cbar=False, cmap=cmap)
    plt.xlabel('Predicted value', fontsize=12)
    plt.ylabel('True value', fontsize=12)
    plt.title(title, fontsize=12, weight='semibold');


def plot_prediction_grid(X_raw, y_raw, preds, class_labels, n_rows=10, n_cols=10):
    _, axes = plt.subplots(n_rows, n_cols, figsize=(12, 12))

    for i, ax in enumerate(axes.flatten()):
        if i >= len(preds):
            break

        ax.imshow(X_raw[i])
        ax.set_xticklabels([])
        ax.set_yticklabels([])

        pred = preds[i]
        act = class_labels[y_raw[i]]
        color = 'white' if pred == act else 'tomato'
        ax.text(0.05, 0.05, pred, color=color, weight='semibold', transform=ax.transAxes)

    plt.show()

Let's build a NN model with three hidden layers.

In [ ]:
#--

# initialize seeds
init_seeds(314)

# prepare the model architecture
mlp1 = Sequential()

#--

# fit and validate the model
#--

These results don't look very good. Let's try to add more neurons to the hidden nodes.

* Supplimentary Resource:
    * [Early-stopping](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping)    
* Interesting link:
    * [What is the difference between `sparse_categorical_crossentropy` and `categorical_crossentropy`?](https://stackoverflow.com/questions/58565394/what-is-the-difference-between-sparse-categorical-crossentropy-and-categorical-c)

In [ ]:
# initialize seeds
init_seeds(314)

mlp2 = Sequential(
    [
        Input(shape=(32, 32, 3)),
        Flatten(),
        Dense(1024, activation='relu'),
        Dense(1024, activation='relu'),
        Dense(64, activation='relu'),
        Dense(10, activation='softmax')
    ], 
    name='mlp_3hidden_v2')

mlp2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

mlp2.fit(X_train, y_train, epochs=5, shuffle=True);

Again, the NN model is not able to achieve high accuracy. We could try more complicated models but there's a better way to improve the model --> CNN!

### Build a Convolutional Neural Network (CNN) model

In [ ]:
# initialize seeds
init_seeds(314)

cnn = Sequential(
    [
        Input(shape=(32, 32, 3)),
        #--
        # Conv2D(...)
        # MaxPooling2D(...)
        # Flatten()
        # Dense(...)
        # Dense(...)
        # Dense(...)
    ],
    name='cnn')

cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

cnn.fit(X_train, y_train, epochs=10, shuffle=True);

In [ ]:
loss, accuracy = cnn.evaluate(X_test, y_test)
print(f'Loss: {loss:.4f}, Accuracy: {accuracy*100:.2f}%')

Further fine-tuning would improve the model accuracy. For now, let's proceed with this model.

* Reflections:
    * [How to avoid overfitting in Deep Learning Neural Networks?](https://machinelearningmastery.com/introduction-to-regularization-to-reduce-overfitting-and-improve-generalization-error/)
    * [Something that bothers me about deep neural nets.](https://www.johndcook.com/blog/2017/10/09/something-that-bothers-me-about-deep-neural-nets/)

In [ ]:
# predicted probabilities for each class
probs = 

In [ ]:
# grab the predictions (predicted labels) from the model
preds = [class_labels[np.argmax(p)] for p in probs]
#--

### Visualize the predictions

In [ ]:
plot_prediction_grid(X_test_raw, y_test_raw, preds, class_labels, n_rows=10, n_cols=10)

## Exercises:

**Exercise 1:** Try different hyper-parameters to improve the model accuracy. 

**Exercise 2:** Capture the "history" of model fitting (i.e., the output of the `model.fit()` function) and plot (1) model accuracy, (2) validation accuracy, (3) model loss, and (4) validation loss, using `matplotlib`. (You can use `epoch` for the x-axis and put `accuracy` (or `loss`) on the y-axis.)